# Download Videos from S3

In [1]:
! pip install boto3
! pip install python-dotenv


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [2]:
import boto3
import os
from dotenv import load_dotenv
import pandas as pd


load_dotenv()

s3_bucket = 'deliberation-lab-recordings-exaptation'
s3_region = "us-east-1"
recordings_folder = 'recordings'

s3 = boto3.client('s3', region_name=s3_region)

In [3]:
# download every recording from the s3 bucket

def list_all_s3_objects(bucket_name, prefix=''):
    paginator = s3.get_paginator('list_objects_v2')
    
    # Create a PageIterator from the Paginator
    page_iterator = paginator.paginate(Bucket=bucket_name, Prefix=prefix)
    
    all_objects = []
    
    for page in page_iterator:
        if 'Contents' in page:
            for obj in page['Contents']:
                all_objects.append(obj)
    
    return all_objects

# # without pagination just get the first 1000 objects
# response = s3.list_objects_v2(Bucket=s3_bucket)
# listings = response['Contents']

listings = list_all_s3_objects(s3_bucket)
len(listings)



2121

In [4]:
listings[-15:]

[{'Key': 'deliberation/20251120_1759_prolifVCP1YM/1763661988604-04041cee-4200-4c31-b391-a4e96ef5bbfa-cam-video-1763661990134',
  'LastModified': datetime.datetime(2025, 11, 20, 18, 6, 31, tzinfo=tzutc()),
  'ETag': '"13948bf42be6ec38cacbb048313afdee-3"',
  'ChecksumAlgorithm': ['CRC64NVME'],
  'ChecksumType': 'FULL_OBJECT',
  'Size': 15586277,
  'StorageClass': 'STANDARD'},
 {'Key': 'deliberation/20251120_1759_prolifVCP1YM/1763661988604-87bdaa6b-7a57-4879-b7e1-e9c28916cad6-cam-audio-1763661990122',
  'LastModified': datetime.datetime(2025, 11, 20, 18, 6, 31, tzinfo=tzutc()),
  'ETag': '"5c264faaa84d07ba99615acd9202f11e-1"',
  'ChecksumAlgorithm': ['CRC64NVME'],
  'ChecksumType': 'FULL_OBJECT',
  'Size': 546551,
  'StorageClass': 'STANDARD'},
 {'Key': 'deliberation/20251120_1759_prolifVCP1YM/1763661988604-87bdaa6b-7a57-4879-b7e1-e9c28916cad6-cam-video-1763661990127',
  'LastModified': datetime.datetime(2025, 11, 20, 18, 6, 31, tzinfo=tzutc()),
  'ETag': '"9c9940df3bbe08b4d18773431eda040

In [7]:
# to_download = [item for item in listings if item["LastModified"] > pd.Timestamp('2025-09-13', tz='UTC') and "pilot" in item["Key"]]
to_download = [item for item in listings if item["LastModified"] > pd.Timestamp('2025-09-13', tz='UTC') ]
len(to_download)

1024

In [8]:
# for every file in the s3 bucket that starts with deliberation/2024
# if the directory it is in doesn't already exist, create a folder in the recordings folder
# if the file iteself doesn't exist, download the file from the s3 bucket to the folder

for obj in to_download:
    if obj['Key'].startswith('deliberation/2025'):
        folder, file = obj['Key'].replace('deliberation/', "").split('/')
        if not os.path.exists(f'{recordings_folder}/{folder}'):
            print(f'Creating folder {folder}')
            os.makedirs(f'{recordings_folder}/{folder}')
        if not os.path.exists(f'{recordings_folder}/{folder}/{file}.webm'):
            print(f'Downloading {obj["Key"]}')
            s3.download_file(s3_bucket, obj['Key'], f'{recordings_folder}/{folder}/{file}.webm')

Creating folder 20250915_1724_prolif6FJEF8
Creating folder 20250915_1724_prolif7NP676
Creating folder 20250915_1724_prolifDZ8FC0
Creating folder 20250915_1724_prolifVRTJT5
Creating folder 20251009_1516_prolif5JYYR9
Creating folder 20251117_1512_prolif1ATHSF
Creating folder 20251117_1839_prolif117DE3
Creating folder 20251117_1839_prolif1HM695
Creating folder 20251117_1839_prolif3ZX52T
Creating folder 20251117_1839_prolif4SKKED
Creating folder 20251117_1839_prolifMCPGPM
Creating folder 20251117_1839_prolifNDK414
Creating folder 20251117_1839_prolifP44228
Creating folder 20251117_1839_prolifPHDRSR
Creating folder 20251117_1839_prolifSE2YQB
Creating folder 20251117_1839_prolifWDWCEZ
Creating folder 20251117_1839_prolifWGB01P
Creating folder 20251118_1823_prolif03KX0C
Creating folder 20251118_1823_prolif1RGRZM
Creating folder 20251118_1823_prolif3MKMWG
Creating folder 20251118_1823_prolif4PPVRD
Creating folder 20251118_1823_prolif5SWQ9E
Creating folder 20251118_1823_prolif60GG1V
Creating fo

In [5]:
# # figure out how much of the discussion was recorded

# data = pd.read_csv("../data/formatted_data.csv")

# for i, participant in data.iterrows():
#     recordingTimes = []
#     for recordingId in participant['recordingIds']:
        


In [6]:
# ! ffprobe recordings/20240920_1732_CT_topMWAGR8/1726855356783-207f4a62-94d6-43e0-844c-2f8442334cb2-cam-audio-1726855372130.webm

In [7]:
# ! ffmpeg -i recordings/20240920_1732_CT_topMWAGR8/1726855356783-207f4a62-94d6-43e0-844c-2f8442334cb2-cam-audio-1726855372130.webm 2>&1 | grep "Duration"

# Measure the quantity of video we have for each respondent
We want to check and see how much video we have, accounting for reloads and technical issues. Ideally, the measure would be the number of unique seconds in which we have at least 1 (10?) frames of video. Not quite sure how to do this yet.

In [8]:
# ! ffmpeg -i recordings/20240920_1732_CT_topMWAGR8/1726855356783-207f4a62-94d6-43e0-844c-2f8442334cb2-cam-audio-1726855372130.webm -vf "showinfo" -f null - 2>&1 | grep showinfo